In [9]:
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.axes_grid1.inset_locator import zoomed_inset_axes, mark_inset

from qiskit import QuantumCircuit, transpile
from qiskit.circuit import Gate, Parameter
from qiskit.circuit.library import (
    RZGate, XGate, RXGate, RYGate, HGate,
    SGate, SdgGate, CXGate, SwapGate,
    RZZGate, PauliEvolutionGate
)
from qiskit.quantum_info import (
    Operator, SparsePauliOp, Statevector, Pauli
)
from qiskit.transpiler import PassManager, CouplingMap
from qiskit.transpiler.passes import BasisTranslator
from qiskit.transpiler.passes.calibration import RZXCalibrationBuilder
from qiskit.dagcircuit import DAGCircuit
from qiskit.converters import circuit_to_dag, dag_to_circuit
from qiskit.circuit.equivalence_library import SessionEquivalenceLibrary
from qiskit.synthesis.evolution import SuzukiTrotter

# Qiskit Nature
from qiskit_nature.second_q.drivers import PySCFDriver
from qiskit_nature.second_q.mappers import ParityMapper, JordanWignerMapper
from qiskit_nature.units import DistanceUnit

# IonQ
from qiskit_ionq import IonQProvider, GPIGate, GPI2Gate, MSGate, ZZGate


In [ ]:

n_qubits = 4
driver = PySCFDriver(
    atom="H 0 0 0; H 0 0 0.742",
    basis="sto3g",
    charge=0,
    spin=0,
    unit=DistanceUnit.ANGSTROM,
)

molecule = driver.run()
mapper = JordanWignerMapper()
cost_h = mapper.map(molecule.hamiltonian.second_q_op())

print("Cost Hamiltonian:", cost_h)
o_driver_h =SparsePauliOp.from_list([
    ("XIII", -1.0),
    ("IXII", -1.0),
    ("IIXI", -1.0),
    ("IIIX", -1.0),
])


driver_scaling = 0.27
driver_h = SparsePauliOp.from_list([
    ("XIII", -1.0*driver_scaling),
    ("IXII", -1.0*driver_scaling),
    ("IIXI", -1.0*driver_scaling),
    ("IIIX", -1.0*driver_scaling),
])




def build_commutator(op_a: SparsePauliOp, op_b: SparsePauliOp) -> SparsePauliOp:
    ab = op_a @ op_b
    ba = op_b @ op_a
    comm_pre= ab - ba
    comm = comm_pre*1j
    return comm.simplify()

def build_commutator2(op_a: SparsePauliOp, op_b: SparsePauliOp) -> SparsePauliOp:
    ab = op_a @ op_b
    ba = op_b @ op_a
    comm= ab - ba
    return comm.simplify()



comm_h=build_commutator(driver_h, cost_h)

a = 0.2
h_cd = a*build_commutator(cost_h, driver_h)

print("Commutator Hamiltonian:", comm_h)

def falqon_layer(qc,cost_h,driver_h,beta_k,h_cd,gamma_k,delta_t):
    #synth = SuzukiTrotter(reps=1)   ,synthesis=synth
    U_c= PauliEvolutionGate(cost_h,delta_t)
    U_d= PauliEvolutionGate(beta_k*driver_h,delta_t)
    U_cd = PauliEvolutionGate(gamma_k*h_cd, delta_t)

    qc.append(U_c, range(qc.num_qubits))
    qc.append(U_d, range(qc.num_qubits))
    qc.append(U_cd, range(qc.num_qubits))



E_set =-1.13728383

def main_loop(qc,cost_h,driver_h,comm_h,delta_t,beta_0,gamma_0,n_steps):
    beta =[beta_0]
    gamma = [gamma_0]
    energies=[]
    H =[]
    
    state = Statevector.from_label("+" * n_qubits)
    for i in range(n_steps):

        
        qc_layer = QuantumCircuit(n_qubits)
        '''
        H=cost_h+beta[i]*driver_h


        h_cd = build_commutator(H, driver_h)
        print("h_cd:", h_cd)
       
        
        '''
        comm_h2 = build_commutator(cost_h, h_cd)
        falqon_layer(qc_layer,cost_h,driver_h,beta[i],h_cd,gamma[i],delta_t)
        qc.compose(qc_layer, inplace=True)
        state = state.evolve(qc_layer) 

        next_beta =-1*state.expectation_value(comm_h).real
        beta.append(next_beta)

        next_gamma =1*state.expectation_value(comm_h2).real
        gamma.append(next_gamma)
        
        energy = state.expectation_value(cost_h).real
        energies.append(energy)
        print(f"第{i + 1}步：能量 = {energy:.8f} Ha| β_{i} = {beta[i]:+.6f} | γ_{i} = {gamma[i]:+.6f}")


        if i >= 1:
            energy_diff = abs(energies[-1] - energies[-2])
            diff_to_target = abs(energies[-1] - E_set)
            if diff_to_target < 1e-3:
            #if  abs(energies[-1] - energies[-2]) < 1e-6:
                print(f" Converged at step {i + 1}!")
                print(f" Energy change ΔE = {energy_diff:.2e} Ha < 1e-6")
                print(f" different from E_set = {diff_to_target:.2e} Ha < 1e-3")
                #print("cost_h:", cost_h )
                #print("hcd", h_cd) 
                break
        
        
    return beta,gamma, energies, qc


def uniform_superposition_circuit(n_qubits=4):
    qc = QuantumCircuit(n_qubits)
    qc.h(range(n_qubits))  # 对所有 qubit 施加 H 门
    return qc

qc_initial= uniform_superposition_circuit()

n_steps = 1000
beta_0 = 0.0
gamma_0 = 0.0
delta_t = 0.03
s_beta,s_gamma, s_energies, final_circuit = main_loop(
    qc=qc_initial,
    cost_h=cost_h,
    driver_h=driver_h,
    comm_h=comm_h,
    delta_t=delta_t,
    beta_0=beta_0,
    gamma_0=gamma_0,
    n_steps=n_steps
)
print("Circuit depth:", final_circuit.depth())
# 生成迭代步列表
iterations = list(range(1, len(s_energies) + 1))
# 创建两个子图
coupling_map = CouplingMap.from_full(4)

provider = IonQProvider()
backend_native = provider.get_backend("simulator", gateset="native")

transpiled_circuit = transpile(final_circuit,coupling_map=coupling_map, backend=backend_native)

for gate, count in transpiled_circuit.count_ops().items():
    print(f"{gate}: {count}")



Cost Hamiltonian: SparsePauliOp(['IIII', 'IIIZ', 'IIZI', 'IIZZ', 'IZII', 'IZIZ', 'ZIII', 'ZIIZ', 'YYYY', 'XXYY', 'YYXX', 'XXXX', 'IZZI', 'ZIZI', 'ZZII'],
              coeffs=[-0.81280876+0.j,  0.17110568+0.j, -0.22250985+0.j,  0.12051037+0.j,
  0.17110568+0.j,  0.16859357+0.j, -0.22250985+0.j,  0.16584097+0.j,
  0.0453306 +0.j,  0.0453306 +0.j,  0.0453306 +0.j,  0.0453306 +0.j,
  0.16584097+0.j,  0.17432084+0.j,  0.12051037+0.j])
Commutator Hamiltonian: SparsePauliOp(['YIII', 'YIIZ', 'ZYYY', 'ZYXX', 'YIZI', 'YZII', 'IYII', 'IYIZ', 'YZYY', 'YZXX', 'IYZI', 'ZYII', 'IIYI', 'IIYZ', 'YYZY', 'XXZY', 'IZYI', 'ZIYI', 'IIIY', 'IIZY', 'IZIY', 'ZIIY', 'YYYZ', 'XXYZ'],
              coeffs=[ 0.12015532+0.j, -0.08955412+0.j,  0.02447852+0.j,  0.02447852+0.j,
 -0.09413326+0.j, -0.0650756 +0.j, -0.09239707+0.j, -0.09104053+0.j,
  0.02447852+0.j,  0.02447852+0.j, -0.08955412+0.j, -0.0650756 +0.j,
  0.12015532+0.j, -0.0650756 +0.j,  0.02447852+0.j,  0.02447852+0.j,
 -0.08955412+0.j, -0.09413326+0.j, -

/tmp/ipykernel_1920/245847552.py:152: DeprecationWarning: The `transpile` function will stop supporting inputs of type `BackendV1` ( ionq_simulator ) in the `backend` parameter in a future release no earlier than 2.0. `BackendV1` is deprecated and implementations should move to `BackendV2`.
  transpiled_circuit = transpile(final_circuit,coupling_map=coupling_map, backend=backend_native)


gpi2: 112016
gpi: 25912
ms: 15048


In [13]:
print("Transpiled circuit depth:", transpiled_circuit.depth())

Transpiled circuit depth: 76210


In [14]:


n_qubits = 4
driver = PySCFDriver(
    atom="H 0 0 0; H 0 0 0.742",
    basis="sto3g",
    charge=0,
    spin=0,
    unit=DistanceUnit.ANGSTROM,
)

molecule = driver.run()
mapper = JordanWignerMapper()
cost_h = mapper.map(molecule.hamiltonian.second_q_op())

print("Cost Hamiltonian:", cost_h)
o_driver_h =SparsePauliOp.from_list([
    ("XIII", -1.0),
    ("IXII", -1.0),
    ("IIXI", -1.0),
    ("IIIX", -1.0),
])


driver_scaling = 0.27
driver_h = SparsePauliOp.from_list([
    ("XIII", -1.0*driver_scaling),
    ("IXII", -1.0*driver_scaling),
    ("IIXI", -1.0*driver_scaling),
    ("IIIX", -1.0*driver_scaling),
])




def build_commutator(op_a: SparsePauliOp, op_b: SparsePauliOp) -> SparsePauliOp:
    ab = op_a @ op_b
    ba = op_b @ op_a
    comm_pre= ab - ba
    comm = comm_pre*1j
    return comm.simplify()

def build_commutator2(op_a: SparsePauliOp, op_b: SparsePauliOp) -> SparsePauliOp:
    ab = op_a @ op_b
    ba = op_b @ op_a
    comm= ab - ba
    return comm.simplify()



comm_h=build_commutator(driver_h, cost_h)

comm_h2 = build_commutator(cost_h, comm_h)
comm_h3 = build_commutator(cost_h, comm_h2)
a=0.2
b=0.2
h_cd = a*comm_h+b*comm_h3


print("Commutator Hamiltonian:", comm_h)

def falqon_layer(qc,cost_h,driver_h,beta_k,h_cd,gamma_k,delta_t):
    #synth = SuzukiTrotter(reps=1)   ,synthesis=synth
    U_c= PauliEvolutionGate(cost_h,delta_t)
    U_d= PauliEvolutionGate(beta_k*driver_h,delta_t)
    U_cd = PauliEvolutionGate(gamma_k*h_cd, delta_t)

    qc.append(U_c, range(qc.num_qubits))
    qc.append(U_d, range(qc.num_qubits))
    qc.append(U_cd, range(qc.num_qubits))



E_set =-1.13728383

def main_loop(qc,cost_h,driver_h,comm_h,delta_t,beta_0,gamma_0,n_steps):
    beta =[beta_0]
    gamma = [gamma_0]
    energies=[]
    H =[]
    
    state = Statevector.from_label("+" * n_qubits)
    for i in range(n_steps):

        
        qc_layer = QuantumCircuit(n_qubits)
        '''
        H=cost_h+beta[i]*driver_h


        h_cd = build_commutator(H, driver_h)
        print("h_cd:", h_cd)
       
        
        '''
        comm_h2 = build_commutator(cost_h, h_cd)
        falqon_layer(qc_layer,cost_h,driver_h,beta[i],h_cd,gamma[i],delta_t)
        qc.compose(qc_layer, inplace=True)
        state = state.evolve(qc_layer) 

        next_beta =-1*state.expectation_value(comm_h).real
        beta.append(next_beta)

        next_gamma =1*state.expectation_value(comm_h2).real
        gamma.append(next_gamma)
        
        energy = state.expectation_value(cost_h).real
        energies.append(energy)
        print(f"第{i + 1}步：能量 = {energy:.8f} Ha| β_{i} = {beta[i]:+.6f} | γ_{i} = {gamma[i]:+.6f}")


        if i >= 1:
            energy_diff = abs(energies[-1] - energies[-2])
            diff_to_target = abs(energies[-1] - E_set)
            if diff_to_target < 1e-3:
            #if  abs(energies[-1] - energies[-2]) < 1e-6:
                print(f" Converged at step {i + 1}!")
                print(f" Energy change ΔE = {energy_diff:.2e} Ha < 1e-6")
                print(f" different from E_set = {diff_to_target:.2e} Ha < 1e-3")
                #print("cost_h:", cost_h )
                #print("hcd", h_cd) 
                break
        
        
    return beta,gamma, energies, qc


def uniform_superposition_circuit(n_qubits=4):
    qc = QuantumCircuit(n_qubits)
    qc.h(range(n_qubits))  # 对所有 qubit 施加 H 门
    return qc

qc_initial= uniform_superposition_circuit()

n_steps = 1000
beta_0 = 0.0
gamma_0 = 0.0
delta_t = 0.03
s_beta,s_gamma, s_energies, final_circuit = main_loop(
    qc=qc_initial,
    cost_h=cost_h,
    driver_h=driver_h,
    comm_h=comm_h,
    delta_t=delta_t,
    beta_0=beta_0,
    gamma_0=gamma_0,
    n_steps=n_steps
)
print("Circuit depth:", final_circuit.depth())
# 生成迭代步列表
iterations = list(range(1, len(s_energies) + 1))
# 创建两个子图
coupling_map = CouplingMap.from_full(4)

provider = IonQProvider()
backend_native = provider.get_backend("simulator", gateset="native")

transpiled_circuit = transpile(final_circuit,coupling_map=coupling_map, backend=backend_native)

for gate, count in transpiled_circuit.count_ops().items():
    print(f"{gate}: {count}")

Cost Hamiltonian: SparsePauliOp(['IIII', 'IIIZ', 'IIZI', 'IIZZ', 'IZII', 'IZIZ', 'ZIII', 'ZIIZ', 'YYYY', 'XXYY', 'YYXX', 'XXXX', 'IZZI', 'ZIZI', 'ZZII'],
              coeffs=[-0.81280876+0.j,  0.17110568+0.j, -0.22250985+0.j,  0.12051037+0.j,
  0.17110568+0.j,  0.16859357+0.j, -0.22250985+0.j,  0.16584097+0.j,
  0.0453306 +0.j,  0.0453306 +0.j,  0.0453306 +0.j,  0.0453306 +0.j,
  0.16584097+0.j,  0.17432084+0.j,  0.12051037+0.j])
Commutator Hamiltonian: SparsePauliOp(['YIII', 'YIIZ', 'ZYYY', 'ZYXX', 'YIZI', 'YZII', 'IYII', 'IYIZ', 'YZYY', 'YZXX', 'IYZI', 'ZYII', 'IIYI', 'IIYZ', 'YYZY', 'XXZY', 'IZYI', 'ZIYI', 'IIIY', 'IIZY', 'IZIY', 'ZIIY', 'YYYZ', 'XXYZ'],
              coeffs=[ 0.12015532+0.j, -0.08955412+0.j,  0.02447852+0.j,  0.02447852+0.j,
 -0.09413326+0.j, -0.0650756 +0.j, -0.09239707+0.j, -0.09104053+0.j,
  0.02447852+0.j,  0.02447852+0.j, -0.08955412+0.j, -0.0650756 +0.j,
  0.12015532+0.j, -0.0650756 +0.j,  0.02447852+0.j,  0.02447852+0.j,
 -0.08955412+0.j, -0.09413326+0.j, -

/tmp/ipykernel_1920/3304191439.py:156: DeprecationWarning: The `transpile` function will stop supporting inputs of type `BackendV1` ( ionq_simulator ) in the `backend` parameter in a future release no earlier than 2.0. `BackendV1` is deprecated and implementations should move to `BackendV2`.
  transpiled_circuit = transpile(final_circuit,coupling_map=coupling_map, backend=backend_native)


gpi2: 378124
gpi: 81710
ms: 53676


In [15]:
print("Transpiled circuit depth:", transpiled_circuit.depth())

Transpiled circuit depth: 272401
